# Phase 7 — Stage A: dictionary seeding

Use a Colab GPU runtime. This stage trains a small self-supervised patch autoencoder, clusters 100,000 16×16 patch embeddings into 4,096 unit-normalized atoms, and records the initial coherence. It deliberately does not save generated/prediction images. Encoder and K-Means++ checkpoints/status are written to Google Drive so an interrupted run can resume by rerunning the training cell. The dictionary artifact, JSON metrics, and downloaded DIV2K data are also persisted in Drive. The setup cell clones the latest `master` when needed and fast-forward updates an existing clone.

DIV2K and Flickr2K support automatic downloads to Drive. DIV2K is restricted to academic research; explicitly accept its terms in the acquisition cell. Flickr2K's EDSR archive source (`https://cv.snu.ac.kr/research/EDSR/Flickr2K.tar`) may be unavailable, in which case place its archive/folder on Drive. Stage A uses both datasets for seeding.

In [ ]:
from pathlib import Path
import subprocess
import sys

from google.colab import drive
drive.mount('/content/drive')

REPO = Path('/content/SparseDict')
DATA_ROOT = Path('/content/drive/MyDrive/SparseDict/datasets')
RUN_DIR = Path('/content/drive/MyDrive/SparseDict/phase7/stage_a')
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('GPU:', subprocess.run(['nvidia-smi'], check=False, capture_output=True, text=True).stdout[:1000])

In [ ]:
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', 'master', 'https://github.com/kalpeshjp89/SparseDict.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'master'], check=True)
sys.path.insert(0, str(REPO))
import torch
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime in Colab before training.')

In [ ]:
# Public downloads go to Drive and are reused on subsequent runs.
# Review DIV2K terms before setting ACCEPT_DIV2K_TERMS=True.
DOWNLOAD_PUBLIC_DATASETS = True
ACCEPT_DIV2K_TERMS = False
# Optional fallbacks: archives already on Drive, shared Drive file IDs, or
# authorized direct HTTPS archive URLs. Fill only what you need.
DRIVE_ARCHIVES = {}  # e.g. {'flickr2k': '/content/drive/MyDrive/datasets/Flickr2K.tar'}
DRIVE_FILE_IDS = {}  # e.g. {'flickr2k': 'Google-Drive-file-id'}
SOURCE_URLS = {}
command = [sys.executable, str(REPO / 'scripts' / 'prepare_phase7_datasets.py'),
           '--root', str(DATA_ROOT), '--dataset', 'div2k', '--dataset', 'flickr2k']
if DOWNLOAD_PUBLIC_DATASETS:
    command.append('--download-public')
if ACCEPT_DIV2K_TERMS:
    command.append('--accept-div2k-license')
for dataset, path in DRIVE_ARCHIVES.items():
    command.extend(['--drive-archive', f'{dataset}={path}'])
for dataset, file_id in DRIVE_FILE_IDS.items():
    command.extend(['--drive-file-id', f'{dataset}={file_id}'])
for dataset, url in SOURCE_URLS.items():
    command.extend(['--source-url', f'{dataset}={url}'])
subprocess.run(command, cwd=REPO, check=True)

In [ ]:
from scripts.prepare_phase7_datasets import resolve_clean_image_dir
IMAGE_DIRS = [resolve_clean_image_dir(name, DATA_ROOT) for name in ('div2k', 'flickr2k')]
print('Stage A image folders:', IMAGE_DIRS)

command = [
    sys.executable, str(REPO / 'scripts' / 'init_dictionaries.py'),
    '--output-dir', str(RUN_DIR), '--patch-count', '100000',
    '--cluster-count', '4096', '--embedding-dim', '64',
    '--encoder-epochs', '3', '--device', 'cuda'
]
for directory in IMAGE_DIRS:
    command.extend(['--image-dir', str(directory)])
print(' '.join(command))
subprocess.run(command, cwd=REPO, check=True)
print('Persistent Stage A artifacts:', sorted(path.name for path in RUN_DIR.iterdir()))